In [1]:
!pip install -q langchain "langchain-community<0.3.28" langchain-huggingface faiss-cpu sentence-transformers
!pip install -q transformers accelerate bitsandbytes gradio
!pip install gradio -q

In [2]:
!pip install -q docx2txt pypdf langchain langchain-community sentence-transformers faiss-cpu


In [3]:
# 1. INSTALL NECESSARY LIBRARIES
# ติดตั้งไลบรารีที่จำเป็นทั้งหมดรวมถึงเวอร์ชันอัปเดตของ LangChain

import torch
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# 2. LOAD DATA: รวมไฟล์ PDF (EN/TH) และ DOCX (TH)
documents = []
file_paths = [
    "/content/sample_data/MLII TOEIC Regis_eng.pdf",
    "/content/sample_data/MLII TOEIC Regis-TH.pdf",
    "/content/sample_data/คำถาม-ตอบ-งานสอบTOEIC.docx"
]

print("--- Start Loading Documents ---")
for path in file_paths:
    if os.path.exists(path):
        print(f"Loading: {path}")
        if path.endswith(".pdf"):
            loader = PyPDFLoader(path)
        elif path.endswith(".docx"):
            loader = Docx2txtLoader(path)
        documents.extend(loader.load())
    else:
        print(f"Warning: File not found at {path}")

# ตรวจสอบความถูกต้องของข้อมูล
if not documents:
    raise FileNotFoundError("CRITICAL ERROR: No documents loaded. Please check your file paths in /content/sample_data/")

# 3. SPLIT DATA: แบ่งข้อความให้มีขนาดเหมาะสมกับการค้นหา
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
texts = text_splitter.split_documents(documents)
print(f"Total text chunks created: {len(texts)}")

# 4. VECTOR STORE: สร้างฐานข้อมูลสำหรับการค้นหา (Embeddings: all-MiniLM-L6-v2)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(texts, embeddings)
print("Vector database created successfully.")

# 5. SETUP OPENTAIGPT-1.0.0-7B-CHAT (Quantized for 4-bit)
model_id = "openthaigpt/openthaigpt-1.0.0-7b-chat"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Downloading and Loading LLM (this may take a few minutes)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    repetition_penalty=1.1
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# 6. RAG CHAIN SETUP: ตั้งค่าระบบการดึงข้อมูลและตอบคำถาม
template = """คุณคือ MLII TOEIC Guide ตอบคำถามเกี่ยวกับการสอบโดยใช้ข้อมูลจาก Context ที่ให้มาเท่านั้น
หากไม่ทราบข้อมูลจากในเอกสาร ให้ตอบว่า "ขออภัย ไม่พบข้อมูลดังกล่าวในระเบียบการ" อย่าพยายามสร้างคำตอบเอง

Context: {context}
Question: {question}

Helpful Answer (ตอบอย่างละเอียดและเป็นลำดับขั้นตอน):"""

# แก้ไข Syntax Error จากเวอร์ชันก่อนหน้า
QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context", "question"], template=template)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

# 7. TEST THE SYSTEM: ทดสอบถามคำถามจริง
print("\n" + "="*60)
print("SYSTEM READY: MLII TOEIC ASSISTANT")
print("="*60)

queries = [
    "ผู้เข้าสอบต้องเตรียมเครื่องเขียนไปเองหรือไม่?",
    "ต้องแสดงหลักฐานอะไรบ้างในวันสอบสำหรับคนไทย?",
    "ห้ามนำสิ่งของใดเข้าห้องสอบบ้าง?"
]

for query in queries:
    print(f"\nUser: {query}")
    response = qa_chain.invoke(query)
    print(f"Assistant: {response['result']}")

--- Start Loading Documents ---
Loading: /content/sample_data/MLII TOEIC Regis_eng.pdf
Loading: /content/sample_data/MLII TOEIC Regis-TH.pdf
Loading: /content/sample_data/คำถาม-ตอบ-งานสอบTOEIC.docx
Total text chunks created: 39


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database created successfully.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...


SYSTEM READY: MLII TOEIC ASSISTANT

User: ผู้เข้าสอบต้องเตรียมเครื่องเขียนไปเองหรือไม่?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: คุณคือ MLII TOEIC Guide ตอบคำถามเกี่ยวกับการสอบโดยใช้ข้อมูลจาก Context ที่ให้มาเท่านั้น
หากไม่ทราบข้อมูลจากในเอกสาร ให้ตอบว่า "ขออภัย ไม่พบข้อมูลดังกล่าวในระเบียบการ" อย่าพยายามสร้างคำตอบเอง

Context: ความสามารถในการใช้ภาษาอังกฤษ (Communicative English) โดยจะต้องมีผลคะแนนตั้งแต่ 785 คะแนน ขึ้นไป รายละเอียดเพิ่มเติมตามประกาศมหาวิทยาลัย
แม่ฟ้าหลวง ลิงก์



12. Q: ผู้สนใจสมัครสอบสามารถสมัครสอบผ่านช่องทางใด

A: สมัครสอบผ่านเว็บไซต์ https://toeic.mfu.ac.th

A: อุปกรณ์อิเล็กทรอนิกส์ และของใช้ส่วนตัว ดังต่อไปนี้






















9.Q: เมื่อชำระเงินค่าสมัครสอบแล้ว ไม่มาสอบตามกำหนด สามารถขอเงินค่าสมัครสอบได้หรือไม่

A: ไม่ได้ เนื่องจากค่าสมัครสอบที่ชำระเรียบร้อยแล้ว จะเป็นค่าใช้จ่ายในการจัดสอบที่ได้ดำเนินการตามแผนไว้แล้ว จึงไม่สามารถขอคืนเงินได้



10.Q: ผลคะแนนสอบมีอายุกี่ปี

A: 2 ปี


11.Q: นักศึกษา มฟล. ที่จะสำเร็จการศึกษา สามารถเข้าสอบ TOEIC เพื่อนำผลคะแนนไปใช้แทนการสอบในรายวิชาภาษาอังกฤษ ได้หรือไม่

A: ได้ นักศึกษาที่จะสำเร็จการศึกษาสามารถใช้ผลสอบTOEIC (Listening and Reading

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: คุณคือ MLII TOEIC Guide ตอบคำถามเกี่ยวกับการสอบโดยใช้ข้อมูลจาก Context ที่ให้มาเท่านั้น
หากไม่ทราบข้อมูลจากในเอกสาร ให้ตอบว่า "ขออภัย ไม่พบข้อมูลดังกล่าวในระเบียบการ" อย่าพยายามสร้างคำตอบเอง

Context: ความสามารถในการใช้ภาษาอังกฤษ (Communicative English) โดยจะต้องมีผลคะแนนตั้งแต่ 785 คะแนน ขึ้นไป รายละเอียดเพิ่มเติมตามประกาศมหาวิทยาลัย
แม่ฟ้าหลวง ลิงก์



12. Q: ผู้สนใจสมัครสอบสามารถสมัครสอบผ่านช่องทางใด

A: สมัครสอบผ่านเว็บไซต์ https://toeic.mfu.ac.th

A: อุปกรณ์อิเล็กทรอนิกส์ และของใช้ส่วนตัว ดังต่อไปนี้






















9.Q: เมื่อชำระเงินค่าสมัครสอบแล้ว ไม่มาสอบตามกำหนด สามารถขอเงินค่าสมัครสอบได้หรือไม่

A: ไม่ได้ เนื่องจากค่าสมัครสอบที่ชำระเรียบร้อยแล้ว จะเป็นค่าใช้จ่ายในการจัดสอบที่ได้ดำเนินการตามแผนไว้แล้ว จึงไม่สามารถขอคืนเงินได้



10.Q: ผลคะแนนสอบมีอายุกี่ปี

A: 2 ปี


11.Q: นักศึกษา มฟล. ที่จะสำเร็จการศึกษา สามารถเข้าสอบ TOEIC เพื่อนำผลคะแนนไปใช้แทนการสอบในรายวิชาภาษาอังกฤษ ได้หรือไม่

A: ได้ นักศึกษาที่จะสำเร็จการศึกษาสามารถใช้ผลสอบTOEIC (Listening and Reading

In [4]:
# 1. INSTALLATION

import torch, os, gc
import gradio as gr
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# 2. CLEAR GPU
def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
clear_vram()

# 3. LOAD DOCUMENTS
documents = []
file_paths = ["/content/sample_data/MLII TOEIC Regis_eng.pdf",
              "/content/sample_data/MLII TOEIC Regis-TH.pdf",
              "/content/sample_data/คำถาม-ตอบ-งานสอบTOEIC.docx"]

for path in file_paths:
    if os.path.exists(path):
        loader = PyPDFLoader(path) if path.endswith(".pdf") else Docx2txtLoader(path)
        documents.extend(loader.load())

# 4. RAG SETUP
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(texts, embeddings)

# 5. LLM SETUP
model_id = "openthaigpt/openthaigpt-1.0.0-7b-chat"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config,
                                             device_map="auto", low_cpu_mem_usage=True)

hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer,
                       max_new_tokens=256, temperature=0.1, repetition_penalty=1.1)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# 6. UPDATED PROMPT (FIXES THE "HI" HALLUCINATION)
template = """You are the MLII TOEIC Guide.
- If the user greets you (e.g., "Hi", "Hello"), respond with: "Hello! I am your MLII TOEIC Assistant. How can I help you with the registration rules today?"
- Otherwise, answer the question strictly using the provided Context.
- If the answer is not in the Context, say "I'm sorry, I don't have that information in the official guidelines."
- Do NOT create practice test questions or multiple-choice options.

Context: {context}
Question: {question}

Helpful Answer:"""

QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context", "question"], template=template)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

# 7. PREDICT FUNCTION WITH MANUAL GREETING CHECK
def predict(message, history):
    clear_vram()
    msg = message.lower().strip()
    # Manual bypass for greetings to prevent hallucination
    if msg in ["hi", "hello", "hey", "สวัสดี"]:
        return "Hello! I am your MLII TOEIC Assistant. I can provide information about registration, ID requirements, and test rules at MFU. How can I help you?"

    response = qa_chain.invoke(message)
    return response["result"]

demo = gr.ChatInterface(fn=predict, title="MLII TOEIC Assistant", description="Ask about MFU TOEIC Rules.")
demo.launch(share=True, debug=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3cf63f86ff3aa0a1e2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3cf63f86ff3aa0a1e2.gradio.live


In [5]:
!pip install streamlit -q

In [6]:
%%writefile app.py
import streamlit as st
import torch, os, gc
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# --- GPU MEMORY MANAGEMENT ---
def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()

# --- INITIALIZE SYSTEM ---
@st.cache_resource
def initialize_system():
    # Folder where you will put your PDFs on the server
    data_folder = "data/"
    documents = []
    if os.path.exists(data_folder):
        for file in os.listdir(data_folder):
            full_path = os.path.join(data_folder, file)
            if file.endswith(".pdf"):
                loader = PyPDFLoader(full_path)
                documents.extend(loader.load())

    if not documents:
        return "Please place PDFs in the 'data' folder."

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    texts = text_splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(texts, embeddings)

    model_id = "openthaigpt/openthaigpt-1.0.0-7b-chat"
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

    hf_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256, temperature=0.1)
    llm = HuggingFacePipeline(pipeline=hf_pipe)

    template = """You are the MLII TOEIC Guide. Answer strictly based on Context.
    Context: {context}
    Question: {question}
    Answer:"""

    QA_PROMPT = PromptTemplate(input_variables=["context", "question"], template=template)
    return RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever(search_kwargs={"k": 2}), chain_type_kwargs={"prompt": QA_PROMPT})

st.title("MLII TOEIC Assistant")
qa_chain = initialize_system()

if prompt := st.chat_input("Ask me anything:"):
    with st.chat_message("user"): st.markdown(prompt)
    with st.chat_message("assistant"):
        clear_vram()
        if prompt.lower() in ["hi", "hello"]:
            res = "Hello! I am your MLII TOEIC Assistant."
        else:
            res = qa_chain.invoke(prompt)["result"]
        st.markdown(res)

Overwriting app.py


In [7]:
# 2. Create the requirements.txt file and PLEASE source code HERE !
%%writefile requirements.txt
streamlit
langchain
langchain-community
langchain-huggingface
langchain-text-splitters
pypdf
faiss-cpu
transformers
accelerate
sentence-transformers

Overwriting requirements.txt


In [8]:
# 3. download all file for streamlit and PLEASE source code HERE !
from google.colab import files

files.download('app.py')
files.download('requirements.txt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>